# Aspect and Polarity

In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
import string

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    AutoModel
)

from sklearn.metrics import classification_report


In [2]:

# =========================================================
# 1. CHECK DEVICE
# =========================================================
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# =========================================================
# 2. LOAD DATA (XML + CSV)
# =========================================================
def xml_to_rows(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    rows = []
    for sentence in root.findall("sentence"):
        sid = sentence.get("id")
        text = sentence.find("text").text

        aspect_terms = sentence.find("aspectTerms")
        if aspect_terms is not None:
            for aspect in aspect_terms.findall("aspectTerm"):
                rows.append({
                    "id": sid,
                    "sentence": text,
                    "aspect": aspect.get("term"),
                    "polarity": aspect.get("polarity")
                })
    return pd.DataFrame(rows)


xml_df = xml_to_rows("./Restaurants_Train_v2.xml")

csv_df = pd.read_csv("./Laptop_Train_v2.csv", encoding="utf-8")
csv_df = csv_df[["id", "Sentence", "Aspect Term", "polarity"]]
csv_df.columns = ["id", "sentence", "aspect", "polarity"]

final_df = pd.concat([csv_df, xml_df], ignore_index=True)

final_df["polarity"] = final_df["polarity"].str.strip().str.upper()
final_df["aspect"] = final_df["aspect"].str.replace('"', '')

final_df = final_df.head(6000)

# =========================================================
# 3. TOKENIZATION + BIOES TAGGING
# =========================================================
def tokenize(text):
    return [
        t.strip(string.punctuation).lower()
        for t in text.split()
        if t.strip(string.punctuation)
    ]

def find_span(tokens, aspect_tokens):
    n, m = len(tokens), len(aspect_tokens)
    for i in range(n - m + 1):
        if tokens[i:i+m] == aspect_tokens:
            return i, i + m - 1
    return -1, -1

def build_tags(tokens, start, end, polarity):
    tags = ["O"] * len(tokens)

    if start == -1:
        return tags

    length = end - start + 1

    if length == 1:
        tags[start] = f"S-{polarity}"
    else:
        tags[start] = f"B-{polarity}"
        for i in range(start + 1, end):
            tags[i] = f"I-{polarity}"
        tags[end] = f"E-{polarity}"

    return tags

def convert(df):
    dataset = []

    for _, row in df.iterrows():
        sentence = row["sentence"]
        aspect = row["aspect"]
        polarity = row["polarity"]

        tokens = tokenize(sentence)
        aspect_tokens = tokenize(aspect)

        start, end = find_span(tokens, aspect_tokens)
        tags = build_tags(tokens, start, end, polarity)

        dataset.append(list(zip(tokens, tags)))

    return dataset

bioes_data = convert(final_df)

# =========================================================
# 4. LABELS
# =========================================================
labels = [
    "O",
    "B-POSITIVE", "I-POSITIVE", "E-POSITIVE", "S-POSITIVE",
    "B-NEGATIVE", "I-NEGATIVE", "E-NEGATIVE", "S-NEGATIVE",
    "B-NEUTRAL", "I-NEUTRAL", "E-NEUTRAL", "S-NEUTRAL",
    "B-CONFLICT", "I-CONFLICT", "E-CONFLICT", "S-CONFLICT"
]

label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}

# =========================================================
# 5. DATASET CREATION
# =========================================================
def prepare_dataset(bioes_data):
    tokens_list = []
    labels_list = []

    for sent in bioes_data:
        tokens_list.append([t for t,l in sent])
        labels_list.append([l for t,l in sent])

    return Dataset.from_dict({
        "tokens": tokens_list,
        "labels": labels_list
    })

dataset = prepare_dataset(bioes_data)
dataset = dataset.train_test_split(test_size=0.2)

# =========================================================
# 6. TOKENIZER
# =========================================================
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True
    )

    labels_batch = []

    for i, word_labels in enumerate(examples["labels"]):
        word_ids = tokenized.word_ids(batch_index=i)

        previous = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous:
                label_ids.append(label2id[word_labels[word_idx]])

            else:
                label_ids.append(label2id[word_labels[word_idx]])

            previous = word_idx

        labels_batch.append(label_ids)

    tokenized["labels"] = labels_batch
    return tokenized

tokenized_dataset = dataset.map(tokenize_and_align, batched=True)

# =========================================================
# 7. BERT + BiLSTM MODEL
# =========================================================
class BERT_BiLSTM(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()

        self.bert = AutoModel.from_pretrained(model_name)

        self.lstm = nn.LSTM(
            input_size=self.bert.config.hidden_size,
            hidden_size=256,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(256 * 2, num_labels)

    def forward(self, input_ids=None, attention_mask=None, labels=None):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        x = outputs.last_hidden_state

        x, _ = self.lstm(x)
        x = self.dropout(x)

        logits = self.classifier(x)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fn(
                logits.view(-1, logits.shape[-1]),
                labels.view(-1)
            )

        return {"loss": loss, "logits": logits}

model = BERT_BiLSTM(model_name, len(labels))

# =========================================================
# 8. TRAINING SETUP
# =========================================================
data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir="./absa_bilstm_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    report_to="none"
)

# =========================================================
# 9. METRICS - FIXED VERSION
# =========================================================
def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)

    # Flatten all predictions and labels
    flat_true_labels = []
    flat_true_preds = []

    for i in range(len(preds)):
        for j in range(len(preds[i])):
            if labels[i][j] != -100:  # Skip padding tokens
                flat_true_labels.append(id2label[labels[i][j]])
                flat_true_preds.append(id2label[preds[i][j]])

    return {
        "report": classification_report(flat_true_labels, flat_true_preds, digits=4)
    }

# =========================================================
# 10. TRAINER
# =========================================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

# =========================================================
# 11. SAVE MODEL
# =========================================================
trainer.save_model("./absa_bilstm_model")
tokenizer.save_pretrained("./absa_bilstm_model")

print("Training completed and model saved successfully!")

CUDA available: True
GPU: NVIDIA GeForce RTX 4080


Map:   0%|          | 0/4800 [00:00<?, ? examples/s]

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Report
1,No log,0.355727,precision recall f1-score support B-CONFLICT 0.0000 0.0000 0.0000 3 B-NEGATIVE 0.0000 0.0000 0.0000 124 B-NEUTRAL 0.0000 0.0000 0.0000 99 B-POSITIVE 0.0000 0.0000 0.0000 251 E-CONFLICT 0.0000 0.0000 0.0000 3 E-NEGATIVE 0.0000 0.0000 0.0000 123 E-NEUTRAL 0.0000 0.0000 0.0000 102 E-POSITIVE 0.0000 0.0000 0.0000 229 I-NEGATIVE 0.0000 0.0000 0.0000 33 I-NEUTRAL 0.0000 0.0000 0.0000 37 I-POSITIVE 0.0000 0.0000 0.0000 122 O 0.9150 0.9991 0.9552 22140 S-CONFLICT 0.0000 0.0000 0.0000 18 S-NEGATIVE 0.0909 0.0035 0.0068 284 S-NEUTRAL 0.0000 0.0000 0.0000 165 S-POSITIVE 0.5169 0.1066 0.1768 572 accuracy 0.9127 24305 macro avg 0.0952 0.0693 0.0712 24305 weighted avg 0.8467 0.9127 0.8744 24305
2,0.442716,0.321428,precision recall f1-score support B-CONFLICT 0.0000 0.0000 0.0000 3 B-NEGATIVE 0.3333 0.0081 0.0157 124 B-NEUTRAL 0.0000 0.0000 0.0000 99 B-POSITIVE 0.0000 0.0000 0.0000 251 E-CONFLICT 0.0000 0.0000 0.0000 3 E-NEGATIVE 0.0000 0.0000 0.0000 123 E-NEUTRAL 0.0000 0.0000 0.0000 102 E-POSITIVE 0.0000 0.0000 0.0000 229 I-NEGATIVE 0.0000 0.0000 0.0000 33 I-NEUTRAL 0.0000 0.0000 0.0000 37 I-POSITIVE 0.0000 0.0000 0.0000 122 O 0.9212 0.9988 0.9584 22140 S-CONFLICT 0.0000 0.0000 0.0000 18 S-NEGATIVE 0.5619 0.2077 0.3033 284 S-NEUTRAL 0.0000 0.0000 0.0000 165 S-POSITIVE 0.6440 0.2150 0.3224 572 accuracy 0.9174 24305 macro avg 0.1538 0.0894 0.1000 24305 weighted avg 0.8626 0.9174 0.8843 24305
3,0.442716,0.305322,precision recall f1-score support B-CONFLICT 0.0000 0.0000 0.0000 3 B-NEGATIVE 0.5217 0.0968 0.1633 124 B-NEUTRAL 0.0000 0.0000 0.0000 99 B-POSITIVE 0.6842 0.0518 0.0963 251 E-CONFLICT 0.0000 0.0000 0.0000 3 E-NEGATIVE 0.6500 0.1057 0.1818 123 E-NEUTRAL 0.0000 0.0000 0.0000 102 E-POSITIVE 0.5778 0.1135 0.1898 229 I-NEGATIVE 0.0000 0.0000 0.0000 33 I-NEUTRAL 0.0000 0.0000 0.0000 37 I-POSITIVE 0.0000 0.0000 0.0000 122 O 0.9284 0.9927 0.9595 22140 S-CONFLICT 0.0000 0.0000 0.0000 18 S-NEGATIVE 0.5519 0.2993 0.3881 284 S-NEUTRAL 0.0000 0.0000 0.0000 165 S-POSITIVE 0.5265 0.3304 0.4060 572 accuracy 0.9182 24305 macro avg 0.2775 0.1244 0.1490 24305 weighted avg 0.8830 0.9182 0.8926 24305
4,0.278629,0.313091,precision recall f1-score support B-CONFLICT 0.0000 0.0000 0.0000 3 B-NEGATIVE 0.5806 0.1452 0.2323 124 B-NEUTRAL 0.0000 0.0000 0.0000 99 B-POSITIVE 0.6078 0.1235 0.2053 251 E-CONFLICT 0.0000 0.0000 0.0000 3 E-NEGATIVE 0.5789 0.1789 0.2733 123 E-NEUTRAL 0.0000 0.0000 0.0000 102 E-POSITIVE 0.4737 0.1572 0.2361 229 I-NEGATIVE 0.0000 0.0000 0.0000 33 I-NEUTRAL 0.0000 0.0000 0.0000 37 I-POSITIVE 0.0000 0.0000 0.0000 122 O 0.9301 0.9909 0.9595 22140 S-CONFLICT 0.0000 0.0000 0.0000 18 S-NEGATIVE 0.5746 0.2711 0.3684 284 S-NEUTRAL 0.0000 0.0000 0.0000 165 S-POSITIVE 0.5106 0.3374 0.4063 572 accuracy 0.9181 24305 macro avg 0.2660 0.1378 0.1676 24305 weighted avg 0.8826 0.9181 0.8948 24305
5,0.238802,0.319234,precision recall f1-score support B-CONFLICT 0.0000 0.0000 0.0000 3 B-NEGATIVE 0.4250 0.2742 0.3333 124 B-NEUTRAL 0.0000 0.0000 0.0000 99 B-POSITIVE 0.3918 0.1514 0.2184 251 E-CONFLICT 0.0000 0.0000 0.0000 3 E-NEGATIVE 0.4366 0.2520 0.3196 123 E-NEUTRAL 0.0000 0.0000 0.0000 102 E-POSITIVE 0.3950 0.2052 0.2701 229 I-NEGATIVE 0.0000 0.0000 0.0000 33 I-NEUTRAL 0.0000 0.0000 0.0000 37 I-POSITIVE 0.0000 0.0000 0.0000 122 O 0.9338 0.9848 0.9586 22140 S-CONFLICT 0.0000 0.0000 0.0000 18 S-NEGATIVE 0.5058 0.3063 0.3816 284 S-NEUTRAL 0.0714 0.0061 0.0112 165 S-POSITIVE 0.5099 0.3601 0.4221 572 accuracy 0.9153 24305 macro avg 0.2293 0.1588 0.1822 24305 weighted avg 0.8812 0.9153 0.8958 24305


c:\Users\lalis\miniconda3\envs\ai_conda\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\lalis\miniconda3\envs\ai_conda\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\lalis\miniconda3\envs\ai_conda\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", 

Training completed and model saved successfully!


# Inference